In [10]:
pip install feedparser

ERROR: Invalid requirement: 'feedparser,': Expected end or semicolon (after name and no valid version specifier)
    feedparser,
              ^


In [9]:
pip install trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.7 MB/s eta 0:00:00


In [12]:
# -*- coding: utf-8 -*-
"""
Improved continuous news + Reddit sentiment scraper.

Key upgrades vs previous version:
- Safe startup: no package installs or drive mounts at import time.
- Works in Colab *and* non-Colab environments.
- Cleaner architecture and explicit config.
- Improved rule-based sentiment with negation handling, confidence shaping,
  and uncertainty penalties.
- FinBERT remains optional (fallback is robust and transparent).
"""

from __future__ import annotations

import gc
import hashlib
import html
import json
import logging
import math
import os
import re
import sys
import time
import traceback
import argparse
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import feedparser
import numpy as np
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup
from dateutil import parser as dateparser


AMSTERDAM_TZ = ZoneInfo("Europe/Amsterdam")
UTC = timezone.utc
REQUEST_TIMEOUT = 20
POLL_INTERVAL_SECONDS = 180
MAX_ARTICLES_PER_FEED = 80
MAX_ITEMS_PER_WINDOW = 1200
BASE_PATH = Path("/content/drive/My Drive/Global News")


def default_output_root() -> Path:
    return BASE_PATH


def in_colab() -> bool:
    return "google.colab" in sys.modules


def maybe_mount_drive() -> None:
    if not in_colab():
        return
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass


def amsterdam_now() -> datetime:
    return datetime.now(AMSTERDAM_TZ)


def floor_to_hour(dt: datetime) -> datetime:
    return dt.astimezone(AMSTERDAM_TZ).replace(minute=0, second=0, microsecond=0)


def parse_datetime(value: Any) -> Optional[datetime]:
    if not value:
        return None
    try:
        dt = value if isinstance(value, datetime) else dateparser.parse(str(value))
        if dt is None:
            return None
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=UTC)
        return dt.astimezone(AMSTERDAM_TZ)
    except Exception:
        return None


def clean_text(text: Optional[str]) -> str:
    if not text:
        return ""
    txt = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    return re.sub(r"\s+", " ", txt).strip()


def normalize_url(url: str) -> str:
    raw = (url or "").strip().split("#")[0]
    if not raw:
        return ""
    # Strip common tracking query parameters for stronger dedupe.
    raw = re.sub(r"([?&](utm_[^=&]+|fbclid|gclid)=[^&#]*)", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"[?&]+$", "", raw)
    return raw


def domain_from_url(url: str) -> str:
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ""


def hash_text(text: str) -> str:
    return hashlib.sha256((text or "").encode("utf-8", errors="ignore")).hexdigest()


class AmsterdamFormatter(logging.Formatter):
    def formatTime(self, record, datefmt=None):
        dt = datetime.fromtimestamp(record.created, tz=AMSTERDAM_TZ)
        return dt.strftime(datefmt) if datefmt else dt.isoformat()


def build_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("news_sentiment_scraper")
    logger.setLevel(logging.INFO)
    logger.handlers = []

    fmt = AmsterdamFormatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S %Z")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger


SOURCE_FEEDS = [
    # -----------------------------
    # Major global / US wires
    # -----------------------------
    {
        "publisher": "Reuters",
        "country": "US",
        "region": "North America",
        "feed_name": "Reuters World",
        "feed_url": "https://news.google.com/rss/search?q=site:reuters.com%20when:1h&hl=en-US&gl=US&ceid=US:en",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "wire",
        "coverage_tags": ["global", "macro", "markets", "companies", "geopolitics"],
    },
    {
        "publisher": "Associated Press",
        "country": "US",
        "region": "North America",
        "feed_name": "AP Top News",
        "feed_url": "https://apnews.com/hub/apf-topnews?output=rss",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "wire",
        "coverage_tags": ["breaking", "politics", "global", "general"],
    },
    {
        "publisher": "Bloomberg",
        "country": "US",
        "region": "North America",
        "feed_name": "Bloomberg Markets",
        "feed_url": "https://news.google.com/rss/search?q=site:bloomberg.com%20(markets%20OR%20economics%20OR%20companies)%20when:1h&hl=en-US&gl=US&ceid=US:en",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "financial_media",
        "coverage_tags": ["markets", "macro", "rates", "fx", "commodities", "companies"],
    },
    {
        "publisher": "CNBC",
        "country": "US",
        "region": "North America",
        "feed_name": "CNBC Latest",
        "feed_url": "https://www.cnbc.com/id/100003114/device/rss/rss.html",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "financial_media",
        "coverage_tags": ["markets", "macro", "companies", "technology", "consumer"],
    },
    {
        "publisher": "MarketWatch",
        "country": "US",
        "region": "North America",
        "feed_name": "MarketWatch Top Stories",
        "feed_url": "https://feeds.content.dowjones.io/public/rss/mw_topstories",
        "source_type": "news",
        "priority": "tier2",
        "publisher_class": "financial_media",
        "coverage_tags": ["markets", "macro", "companies", "rates"],
    },
    {
        "publisher": "Yahoo Finance",
        "country": "US",
        "region": "North America",
        "feed_name": "Yahoo Finance News",
        "feed_url": "https://news.google.com/rss/search?q=site:finance.yahoo.com%20when:1h&hl=en-US&gl=US&ceid=US:en",
        "source_type": "news",
        "priority": "tier2",
        "publisher_class": "financial_media",
        "coverage_tags": ["markets", "companies", "technology", "crypto"],
    },

    # -----------------------------
    # Major UK / European publishers
    # -----------------------------
    {
        "publisher": "BBC",
        "country": "UK",
        "region": "Europe",
        "feed_name": "BBC Front",
        "feed_url": "http://feeds.bbci.co.uk/news/rss.xml",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "broadcaster",
        "coverage_tags": ["global", "politics", "general", "business"],
    },
    {
        "publisher": "BBC",
        "country": "UK",
        "region": "Europe",
        "feed_name": "BBC Business",
        "feed_url": "http://feeds.bbci.co.uk/news/business/rss.xml",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "broadcaster",
        "coverage_tags": ["business", "macro", "markets", "companies"],
    },
    {
        "publisher": "The Guardian",
        "country": "UK",
        "region": "Europe",
        "feed_name": "Guardian World",
        "feed_url": "https://www.theguardian.com/world/rss",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "newspaper",
        "coverage_tags": ["global", "politics", "geopolitics", "general"],
    },
    {
        "publisher": "The Guardian",
        "country": "UK",
        "region": "Europe",
        "feed_name": "Guardian Business",
        "feed_url": "https://www.theguardian.com/uk/business/rss",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "newspaper",
        "coverage_tags": ["business", "macro", "companies", "markets"],
    },
    {
        "publisher": "Financial Times",
        "country": "UK",
        "region": "Europe",
        "feed_name": "FT Global",
        "feed_url": "https://news.google.com/rss/search?q=site:ft.com%20when:1h&hl=en-GB&gl=GB&ceid=GB:en",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "financial_media",
        "coverage_tags": ["macro", "markets", "companies", "policy", "geopolitics"],
    },
    {
        "publisher": "Sky News",
        "country": "UK",
        "region": "Europe",
        "feed_name": "Sky News Home",
        "feed_url": "https://feeds.skynews.com/feeds/rss/home.xml",
        "source_type": "news",
        "priority": "tier2",
        "publisher_class": "broadcaster",
        "coverage_tags": ["breaking", "general", "politics", "world"],
    },

    # -----------------------------
    # International public-interest / policy / economics
    # -----------------------------
    {
        "publisher": "Politico Europe",
        "country": "EU",
        "region": "Europe",
        "feed_name": "Politico Europe",
        "feed_url": "https://www.politico.eu/rss/politics-news/",
        "source_type": "news",
        "priority": "tier2",
        "publisher_class": "policy_media",
        "coverage_tags": ["policy", "europe", "regulation", "politics"],
    },
    {
        "publisher": "Euronews",
        "country": "EU",
        "region": "Europe",
        "feed_name": "Euronews Business",
        "feed_url": "https://www.euronews.com/rss?level=theme&name=business",
        "source_type": "news",
        "priority": "tier2",
        "publisher_class": "broadcaster",
        "coverage_tags": ["europe", "business", "macro", "policy"],
    },
    {
        "publisher": "Al Jazeera",
        "country": "Qatar",
        "region": "Middle East",
        "feed_name": "Al Jazeera News",
        "feed_url": "https://www.aljazeera.com/xml/rss/all.xml",
        "source_type": "news",
        "priority": "tier2",
        "publisher_class": "broadcaster",
        "coverage_tags": ["global", "geopolitics", "middle_east", "breaking"],
    },

    # -----------------------------
    # Macro / central-bank / official releases via news proxies
    # -----------------------------
    {
        "publisher": "Federal Reserve",
        "country": "US",
        "region": "North America",
        "feed_name": "Fed News",
        "feed_url": "https://news.google.com/rss/search?q=Federal%20Reserve%20when:1h&hl=en-US&gl=US&ceid=US:en",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "official",
        "coverage_tags": ["macro", "rates", "policy", "central_banks"],
    },
    {
        "publisher": "ECB",
        "country": "EU",
        "region": "Europe",
        "feed_name": "ECB News",
        "feed_url": "https://news.google.com/rss/search?q=European%20Central%20Bank%20when:1h&hl=en-GB&gl=GB&ceid=GB:en",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "official",
        "coverage_tags": ["macro", "rates", "policy", "central_banks"],
    },
    {
        "publisher": "Bank of England",
        "country": "UK",
        "region": "Europe",
        "feed_name": "BoE News",
        "feed_url": "https://news.google.com/rss/search?q=Bank%20of%20England%20when:1h&hl=en-GB&gl=GB&ceid=GB:en",
        "source_type": "news",
        "priority": "tier1",
        "publisher_class": "official",
        "coverage_tags": ["macro", "rates", "policy", "central_banks"],
    },
]

REDDIT_SOURCES = [
    # -----------------------------
    # Broad news flow
    # -----------------------------
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/news",
        "subreddit": "news",
        "feed_url": "https://www.reddit.com/r/news/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "general_news",
        "coverage_tags": ["breaking", "general", "us_news"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/worldnews",
        "subreddit": "worldnews",
        "feed_url": "https://www.reddit.com/r/worldnews/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "global_news",
        "coverage_tags": ["world", "geopolitics", "global"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/politics",
        "subreddit": "politics",
        "feed_url": "https://www.reddit.com/r/politics/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "politics",
        "coverage_tags": ["us_politics", "policy", "elections"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/unitedkingdom",
        "subreddit": "unitedkingdom",
        "feed_url": "https://www.reddit.com/r/unitedkingdom/new/.rss",
        "source_type": "reddit",
        "priority": "tier2",
        "community_type": "regional_news",
        "coverage_tags": ["uk_news", "uk_politics", "regional"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/ukpolitics",
        "subreddit": "ukpolitics",
        "feed_url": "https://www.reddit.com/r/ukpolitics/new/.rss",
        "source_type": "reddit",
        "priority": "tier2",
        "community_type": "politics",
        "coverage_tags": ["uk_politics", "policy", "government"],
    },

    # -----------------------------
    # Finance / markets / macro
    # -----------------------------
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/stocks",
        "subreddit": "stocks",
        "feed_url": "https://www.reddit.com/r/stocks/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "equities",
        "coverage_tags": ["equities", "markets", "single_stocks"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/investing",
        "subreddit": "investing",
        "feed_url": "https://www.reddit.com/r/investing/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "multi_asset_investing",
        "coverage_tags": ["markets", "macro", "portfolio", "equities"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/economics",
        "subreddit": "economics",
        "feed_url": "https://www.reddit.com/r/economics/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "macro",
        "coverage_tags": ["macro", "economics", "policy", "rates"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/business",
        "subreddit": "business",
        "feed_url": "https://www.reddit.com/r/business/new/.rss",
        "source_type": "reddit",
        "priority": "tier1",
        "community_type": "business_news",
        "coverage_tags": ["companies", "earnings", "corporate", "industry"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/finance",
        "subreddit": "finance",
        "feed_url": "https://www.reddit.com/r/finance/new/.rss",
        "source_type": "reddit",
        "priority": "tier2",
        "community_type": "finance",
        "coverage_tags": ["finance", "macro", "markets"],
    },

    # -----------------------------
    # Specialized / optional higher-noise but useful
    # -----------------------------
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/technology",
        "subreddit": "technology",
        "feed_url": "https://www.reddit.com/r/technology/new/.rss",
        "source_type": "reddit",
        "priority": "tier2",
        "community_type": "technology_news",
        "coverage_tags": ["technology", "ai", "internet", "platforms"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/energy",
        "subreddit": "energy",
        "feed_url": "https://www.reddit.com/r/energy/new/.rss",
        "source_type": "reddit",
        "priority": "tier2",
        "community_type": "sector_news",
        "coverage_tags": ["energy", "oil", "gas", "utilities", "transition"],
    },
    {
        "publisher": "Reddit",
        "country": "Global",
        "region": "Global",
        "feed_name": "r/geopolitics",
        "subreddit": "geopolitics",
        "feed_url": "https://www.reddit.com/r/geopolitics/new/.rss",
        "source_type": "reddit",
        "priority": "tier2",
        "community_type": "geopolitics",
        "coverage_tags": ["geopolitics", "security", "war", "diplomacy"],
    },
]

TOPIC_RULES = {
    "markets": [
        "stock", "stocks", "equity", "equities", "index", "indices", "futures", "options",
        "nasdaq", "s&p", "dow", "ftse", "stoxx", "nikkei", "hang seng", "volatility", "vix",
        "drawdown", "rally", "selloff", "correction", "valuation", "multiple expansion",
        "risk-on", "risk off", "market breadth", "trading volume", "liquidity"
    ],
    "macro": [
        "inflation", "cpi", "ppi", "gdp", "pmi", "ism", "unemployment", "payrolls",
        "labor market", "consumer spending", "retail sales", "housing starts", "industrial production",
        "recession", "soft landing", "stagflation", "fiscal policy", "economic growth", "productivity",
        "deflation", "disinflation"
    ],
    "central_banks": [
        "fed", "federal reserve", "ecb", "european central bank", "bank of england", "boe",
        "boj", "bank of japan", "pboc", "rba", "snb", "rate decision", "interest rate",
        "policy rate", "hawkish", "dovish", "quantitative tightening", "quantitative easing",
        "balance sheet", "forward guidance"
    ],
    "rates_fx": [
        "treasury yield", "bond yield", "yield curve", "2-year yield", "10-year yield",
        "real yield", "dollar", "dxy", "eurusd", "usdjpy", "fx", "foreign exchange",
        "currency intervention", "devaluation", "sterling", "yen", "euro"
    ],
    "credit": [
        "credit spread", "high yield", "investment grade", "default", "distressed debt",
        "leveraged loan", "private credit", "bank lending", "delinquency", "refinancing risk"
    ],
    "commodities": [
        "oil", "crude", "wti", "brent", "natural gas", "lng", "gold", "silver", "copper",
        "iron ore", "aluminum", "nickel", "wheat", "corn", "soybeans", "commodity prices",
        "opec", "mine output", "refinery", "inventory draw"
    ],
    "energy": [
        "energy security", "power grid", "electricity prices", "utilities", "renewables",
        "solar", "wind", "nuclear", "battery storage", "pipeline", "oilfield", "offshore drilling",
        "energy transition", "capacity", "outage"
    ],
    "geopolitics": [
        "war", "conflict", "missile", "drone strike", "troop", "ceasefire", "sanctions",
        "export controls", "tariff", "trade war", "embargo", "border", "diplomacy",
        "summit", "peace talks", "military", "nato", "china tensions", "taiwan", "middle east"
    ],
    "politics_policy": [
        "election", "parliament", "congress", "senate", "white house", "downing street",
        "prime minister", "president", "cabinet", "regulation", "antitrust", "legislation",
        "executive order", "budget", "tax reform", "industrial policy", "subsidy", "stimulus",
        "government shutdown"
    ],
    "companies": [
        "earnings", "revenue", "ebitda", "guidance", "margin", "free cash flow", "buyback",
        "dividend", "merger", "acquisition", "takeover", "layoffs", "restructuring", "spinoff",
        "ipo", "secondary offering", "profit warning", "capital expenditure", "backlog",
        "order book", "inventory overhang", "demand slowdown"
    ],
    "banks": [
        "bank", "lender", "deposit flight", "capital ratio", "tier 1 capital", "net interest margin",
        "loan loss provision", "liquidity coverage ratio", "commercial real estate exposure"
    ],
    "technology": [
        "ai", "artificial intelligence", "semiconductor", "chip", "gpu", "data center", "cloud",
        "software", "cybersecurity", "openai", "microsoft", "nvidia", "amd", "broadcom",
        "foundry", "fab", "hyperscaler", "enterprise software", "saas"
    ],
    "healthcare": [
        "drug trial", "fda", "ema approval", "biotech", "pharma", "medical device",
        "hospital", "health insurer", "patent challenge", "clinical results"
    ],
    "consumer": [
        "consumer confidence", "retail", "e-commerce", "luxury", "travel demand", "airline",
        "hotel", "restaurant", "foot traffic", "pricing power", "inventory build"
    ],
    "real_estate": [
        "commercial real estate", "office vacancy", "home prices", "mortgage rates",
        "housing affordability", "construction", "property developer", "reit"
    ],
    "crypto": [
        "bitcoin", "ethereum", "crypto", "stablecoin", "etf inflows", "mining difficulty",
        "on-chain", "exchange outflows", "token unlock", "defi"
    ],
    "supply_chain": [
        "shipping rates", "container rates", "logistics", "port congestion", "inventory cycle",
        "supplier disruption", "freight", "red sea", "semiconductor supply chain"
    ],
}

SECTOR_TOPIC_MAP = {
    "technology": "Technology",
    "markets": "Multi-Asset / Market Structure",
    "macro": "Macro / Rates",
    "central_banks": "Macro / Rates",
    "rates_fx": "Macro / Rates",
    "credit": "Financials / Credit",
    "banks": "Financials / Credit",
    "companies": "Corporate / Cross-Sector",
    "consumer": "Consumer",
    "healthcare": "Healthcare",
    "real_estate": "Real Estate",
    "commodities": "Commodities / Materials",
    "energy": "Energy",
    "geopolitics": "Geopolitics / Sovereign Risk",
    "politics_policy": "Government / Regulation",
    "crypto": "Digital Assets",
    "supply_chain": "Industrials / Logistics",
}

SECTOR_TICKERS = {
    "Technology": [
        ("AAPL", "Apple"),
        ("MSFT", "Microsoft"),
        ("NVDA", "NVIDIA"),
        ("AMD", "AMD"),
        ("AVGO", "Broadcom"),
        ("TSM", "TSMC"),
    ],
    "Multi-Asset / Market Structure": [
        ("SPY", "SPDR S&P 500 ETF"),
        ("QQQ", "Invesco QQQ"),
        ("IWM", "iShares Russell 2000 ETF"),
        ("VXX", "iPath VIX ETN"),
        ("CME", "CME Group"),
    ],
    "Macro / Rates": [
        ("TLT", "iShares 20+ Year Treasury Bond ETF"),
        ("IEF", "iShares 7-10 Year Treasury Bond ETF"),
        ("UUP", "Invesco DB US Dollar Index Bullish Fund"),
        ("FXE", "Invesco CurrencyShares Euro Trust"),
        ("GLD", "SPDR Gold Shares"),
    ],
    "Financials / Credit": [
        ("JPM", "JPMorgan"),
        ("GS", "Goldman Sachs"),
        ("MS", "Morgan Stanley"),
        ("BAC", "Bank of America"),
        ("KRE", "SPDR S&P Regional Banking ETF"),
        ("HYG", "iShares iBoxx High Yield Corporate Bond ETF"),
    ],
    "Corporate / Cross-Sector": [
        ("GE", "GE Aerospace"),
        ("CAT", "Caterpillar"),
        ("HON", "Honeywell"),
        ("UNP", "Union Pacific"),
        ("MMM", "3M"),
    ],
    "Consumer": [
        ("AMZN", "Amazon"),
        ("WMT", "Walmart"),
        ("COST", "Costco"),
        ("MCD", "McDonald's"),
        ("BKNG", "Booking Holdings"),
    ],
    "Healthcare": [
        ("LLY", "Eli Lilly"),
        ("JNJ", "Johnson & Johnson"),
        ("MRK", "Merck"),
        ("UNH", "UnitedHealth"),
        ("ISRG", "Intuitive Surgical"),
    ],
    "Real Estate": [
        ("VNQ", "Vanguard Real Estate ETF"),
        ("AMT", "American Tower"),
        ("PLD", "Prologis"),
        ("SPG", "Simon Property Group"),
        ("CBRE", "CBRE Group"),
    ],
    "Commodities / Materials": [
        ("XLB", "Materials Select Sector SPDR Fund"),
        ("FCX", "Freeport-McMoRan"),
        ("NEM", "Newmont"),
        ("AA", "Alcoa"),
        ("SCCO", "Southern Copper"),
    ],
    "Energy": [
        ("XOM", "Exxon Mobil"),
        ("CVX", "Chevron"),
        ("COP", "ConocoPhillips"),
        ("SLB", "Schlumberger"),
        ("XLE", "Energy Select Sector SPDR Fund"),
    ],
    "Geopolitics / Sovereign Risk": [
        ("GLD", "SPDR Gold Shares"),
        ("UUP", "Invesco DB US Dollar Index Bullish Fund"),
        ("ITA", "iShares U.S. Aerospace & Defense ETF"),
        ("XLE", "Energy Select Sector SPDR Fund"),
    ],
    "Government / Regulation": [
        ("XLI", "Industrial Select Sector SPDR Fund"),
        ("XLV", "Health Care Select Sector SPDR Fund"),
        ("XLF", "Financial Select Sector SPDR Fund"),
        ("XLC", "Communication Services Select Sector SPDR Fund"),
    ],
    "Digital Assets": [
        ("IBIT", "iShares Bitcoin Trust"),
        ("BITO", "ProShares Bitcoin Strategy ETF"),
        ("COIN", "Coinbase"),
        ("MSTR", "MicroStrategy"),
    ],
    "Industrials / Logistics": [
        ("FDX", "FedEx"),
        ("UPS", "UPS"),
        ("UNP", "Union Pacific"),
        ("CSX", "CSX"),
        ("EXPD", "Expeditors"),
    ],
}

POSITIVE_TERMS = {
    # Growth / earnings / activity
    "beat", "beats", "beating expectations", "above expectations", "strong demand",
    "robust demand", "sales growth", "revenue growth", "profit growth", "margin expansion",
    "record profit", "record revenue", "upside surprise", "guidance raised", "raises guidance",
    "accelerating growth", "strong outlook", "order growth", "backlog growth", "cash flow strength",

    # Macro / policy positive
    "soft landing", "disinflation", "inflation cools", "inflation eases", "lower inflation",
    "rate cuts", "policy easing", "dovish shift", "stimulus", "fiscal support", "productivity gains",
    "resilient economy", "labor market resilience",

    # Market positive
    "rally", "rebound", "recovery", "multiple expansion", "risk-on", "spread tightening",
    "yield declines", "lower yields", "improving liquidity", "credit improvement",

    # Geopolitical / supply positive
    "ceasefire", "de-escalation", "deal reached", "peace talks progress", "sanctions relief",
    "supply recovery", "inventory normalization", "shipping disruption eases",

    # Corporate actions
    "upgrade", "buyback", "dividend increase", "strategic partnership", "cost discipline",
    "efficiency gains", "turnaround", "successful restructuring", "asset sale"
}

NEGATIVE_TERMS = {
    # Earnings / corporate negative
    "miss", "misses", "below expectations", "weak demand", "demand slowdown", "profit warning",
    "guidance cut", "cuts guidance", "margin compression", "inventory glut", "inventory overhang",
    "sales slump", "revenue decline", "earnings decline", "cash burn", "write-down",
    "impairment", "restructuring charges", "layoffs", "default", "bankruptcy", "fraud",

    # Macro / policy negative
    "inflation accelerates", "sticky inflation", "hot inflation", "rate hike", "higher for longer",
    "hawkish shift", "recession risk", "hard landing", "stagflation", "weak payrolls",
    "rising unemployment", "fiscal drag", "policy uncertainty",

    # Market negative
    "selloff", "slump", "drawdown", "correction", "liquidity stress", "spread widening",
    "higher yields", "yield spike", "volatility surge", "risk-off", "capital outflows",

    # Geopolitical / supply negative
    "war", "escalation", "missile strike", "drone attack", "sanctions", "tariff increase",
    "trade war", "embargo", "supply disruption", "port congestion", "shipping disruption",
    "export controls", "cyberattack",

    # Financial stability negative
    "deposit flight", "credit losses", "liquidity crisis", "bank stress", "downgrade",
    "rating cut", "delinquency rise", "commercial real estate stress"
}

NEGATIONS = {
    "no", "not", "never", "without", "hardly", "rarely", "neither", "nor", "isn't", "wasn't",
    "aren't", "don't", "doesn't", "didn't", "can't", "couldn't", "won't", "wouldn't"
}

UNCERTAINTY_TERMS = {
    "may", "might", "could", "unclear", "uncertainty", "rumor", "rumour", "reportedly",
    "possible", "possibly", "suggests", "appears", "indicates", "potential", "tentative",
    "unconfirmed", "if", "contingent", "preliminary", "sources say", "according to sources",
    "discussion", "considering", "weighing", "exploring", "expected to", "likely", "unlikely"
}


@dataclass
class ArticleRecord:
    article_id: str
    canonical_id: str
    source_type: str
    publisher: str
    country: str
    origin_country: str
    feed_name: str
    subreddit: str
    source_feed_url: str
    url: str
    domain: str
    title: str
    summary: str
    body: str
    authors: str
    published_at: Optional[str]
    first_seen_at: str
    final_sentiment_label: str
    final_sentiment_score: float
    market_sentiment_label: str
    market_sentiment_score: float
    sentiment_model_used: str
    sentiment_model_note: str
    confidence_score: float
    topics: List[str]


@dataclass
class FeedSource:
    publisher: str
    country: str
    feed_name: str
    feed_url: str
    source_type: str
    subreddit: str = ""


class StateManager:
    def __init__(self, state_dir: Path):
        self.seen_file = state_dir / "seen_urls.json"
        self.history_file = state_dir / "run_history.json"
        self.completed_file = state_dir / "completed_windows.json"
        self.hourly_prediction_file = state_dir / "hourly_sector_predictions.json"
        self.daily_prediction_file = state_dir / "daily_sector_predictions.json"
        self.seen_urls: Dict[str, str] = self._load(self.seen_file, {})
        self.run_history: List[Dict[str, Any]] = self._load(self.history_file, [])
        self.completed_windows = set(self._load(self.completed_file, []))
        self.hourly_predictions: Dict[str, Any] = self._load(self.hourly_prediction_file, {})
        self.daily_predictions: Dict[str, Any] = self._load(self.daily_prediction_file, {})

    @staticmethod
    def _load(path: Path, default: Any) -> Any:
        if path.exists():
            try:
                return json.loads(path.read_text(encoding="utf-8"))
            except Exception:
                return default
        return default

    @staticmethod
    def _save(path: Path, payload: Any) -> None:
        tmp = path.with_suffix(path.suffix + ".tmp")
        tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        tmp.replace(path)

    def persist(self) -> None:
        self._save(self.seen_file, self.seen_urls)
        self._save(self.history_file, self.run_history[-2000:])
        self._save(self.completed_file, sorted(self.completed_windows))
        self._save(self.hourly_prediction_file, self.hourly_predictions)
        self._save(self.daily_prediction_file, self.daily_predictions)


class SentimentEngine:
    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.finbert_model = None
        self.social_model = None
        try:
            from transformers import pipeline

            self.finbert_model = pipeline(
                task="text-classification",
                model="ProsusAI/finbert",
                tokenizer="ProsusAI/finbert",
                truncation=True,
                max_length=512,
                device=-1,
            )
            self.logger.info("FinBERT loaded (news/finance path).")
        except Exception as exc:
            self.logger.warning("FinBERT unavailable, using rule-based sentiment for finance/news: %s", exc)

        try:
            from transformers import pipeline

            model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
            self.social_model = pipeline(
                task="text-classification",
                model=model_name,
                tokenizer=model_name,
                truncation=True,
                max_length=512,
                device=-1,
            )
            self.logger.info("Twitter RoBERTa loaded (reddit/social path).")
        except Exception as exc:
            self.logger.warning("Twitter RoBERTa unavailable, using rule-based sentiment for reddit/social: %s", exc)

    def _rule_score(self, text: str) -> Tuple[str, float, float]:
        lowered = text.lower()
        tokens = re.findall(r"[a-z']+", lowered)
        if not tokens:
            return "neutral", 0.5, 0.0

        raw = 0.0
        for term in POSITIVE_TERMS:
            if re.search(rf"\\b{re.escape(term)}\\b", lowered):
                raw += 1.8
        for term in NEGATIVE_TERMS:
            if re.search(rf"\\b{re.escape(term)}\\b", lowered):
                raw -= 1.8

        # Negation handling around common polarity words.
        for i, t in enumerate(tokens):
            if t in NEGATIONS and i + 1 < len(tokens):
                nxt = tokens[i + 1]
                if nxt in {"good", "positive", "growth", "beat", "gain"}:
                    raw -= 1.2
                if nxt in {"bad", "negative", "loss", "drop", "slump"}:
                    raw += 1.2

        uncertainty_hits = sum(1 for w in tokens if w in UNCERTAINTY_TERMS)
        raw /= max(2.0, math.sqrt(len(tokens)))
        raw *= max(0.65, 1 - uncertainty_hits * 0.06)

        if raw > 0.12:
            label = "positive"
        elif raw < -0.12:
            label = "negative"
        else:
            label = "neutral"
        confidence = float(np.clip(0.52 + abs(raw), 0.0, 0.98))
        return label, confidence, float(raw)

    def score(self, text: str, topics: List[str], source_type: str = "news") -> Dict[str, Any]:
        text = clean_text(text)
        if not text:
            return {
                "label": "neutral",
                "confidence": 0.5,
                "raw": 0.0,
                "market_raw": 0.0,
                "market_label": "neutral",
                "model_used": "none",
                "model_note": "empty_text",
            }

        use_social_model = source_type == "reddit" and self.social_model is not None
        use_finbert_model = source_type != "reddit" and self.finbert_model is not None

        model_used = "rule_based"
        model_note = "rule_fallback"
        if use_social_model or use_finbert_model:
            try:
                pipeline_model = self.social_model if use_social_model else self.finbert_model
                out = pipeline_model(text[:3000])[0]
                lbl = out["label"].lower()
                if "positive" in lbl or "label_2" in lbl or lbl.endswith("2"):
                    label, raw = "positive", float(out["score"])
                elif "negative" in lbl or "label_0" in lbl or lbl.endswith("0"):
                    label, raw = "negative", -float(out["score"])
                else:
                    label, raw = "neutral", 0.0
                confidence = float(out["score"])
                model_used = "twitter_roberta" if use_social_model else "finbert"
                model_note = f"transformer_label={out['label']}"
            except Exception:
                label, confidence, raw = self._rule_score(text)
                model_used = "rule_based"
                model_note = "transformer_error_rule_fallback"
        else:
            label, confidence, raw = self._rule_score(text)
            model_used = "rule_based"
            model_note = "transformer_unavailable_rule_fallback"

        market_adj = 0.0
        ltxt = text.lower()
        if "macro" in topics and any(p in ltxt for p in ["inflation cools", "lower inflation", "rate cuts"]):
            market_adj += 0.55
        if "macro" in topics and any(p in ltxt for p in ["hot inflation", "rate hike", "higher for longer"]):
            market_adj -= 0.55
        if "geopolitics" in topics and any(p in ltxt for p in ["war", "missile", "escalation"]):
            market_adj -= 0.65
        if "companies" in topics and any(p in ltxt for p in ["beats", "raises guidance"]):
            market_adj += 0.35

        market_raw = float(np.clip(0.65 * raw + 0.35 * market_adj, -1.5, 1.5))
        if market_raw > 0.12:
            market_label = "positive"
        elif market_raw < -0.12:
            market_label = "negative"
        else:
            market_label = "neutral"

        return {
            "label": label,
            "confidence": confidence,
            "raw": float(raw),
            "market_raw": market_raw,
            "market_label": market_label,
            "model_used": model_used,
            "model_note": model_note,
        }


class Scraper:
    def __init__(self, output_root: Path, sources: Optional[List[FeedSource]] = None):
        self.output_root = output_root
        for p in ["logs", "state", "Report", "Data"]:
            (output_root / p).mkdir(parents=True, exist_ok=True)

        self.logger = build_logger(output_root / "logs" / "aggregator.log")
        self.state = StateManager(output_root / "state")
        self.sentiment = SentimentEngine(self.logger)
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; NewsSentimentScraper/2.0)"})

        self.active_start = floor_to_hour(amsterdam_now())
        self.active_end = self.active_start + timedelta(hours=1)
        self.buffer: List[Dict[str, Any]] = []
        self.buffer_urls: set[str] = set()
        self.source_stats: Dict[str, Dict[str, int]] = {}
        self.sources = sources or default_sources()

    def _day_dirs(self, reference_dt: datetime) -> Tuple[Path, Path]:
        day_str = reference_dt.astimezone(AMSTERDAM_TZ).strftime("%d-%m-%Y")
        dashboard_dir = self.output_root / "Report" / day_str
        data_dir = self.output_root / "Data" / day_str
        dashboard_dir.mkdir(parents=True, exist_ok=True)
        data_dir.mkdir(parents=True, exist_ok=True)
        return dashboard_dir, data_dir

    def _in_window(self, dt: Optional[datetime]) -> bool:
        return bool(dt and self.active_start <= dt < self.active_end)

    def _discover(self) -> List[Dict[str, Any]]:
        candidates: List[Dict[str, Any]] = []
        for source in self.sources:
            source_cfg = asdict(source)
            source_name = source_cfg["feed_name"]
            self.source_stats.setdefault(source_name, {"fetched": 0, "accepted": 0, "errors": 0})
            try:
                parsed = feedparser.parse(source_cfg["feed_url"])
                for entry in parsed.entries[:MAX_ARTICLES_PER_FEED]:
                    self.source_stats[source_name]["fetched"] += 1
                    url = normalize_url(getattr(entry, "link", "") or getattr(entry, "id", ""))
                    if not url or url in self.state.seen_urls:
                        continue
                    published_at = parse_datetime(
                        getattr(entry, "published", None)
                        or getattr(entry, "updated", None)
                        or getattr(entry, "pubDate", None)
                    )
                    if not self._in_window(published_at):
                        continue
                    candidates.append(
                        {
                            "source_type": source_cfg["source_type"],
                            "publisher": source_cfg["publisher"],
                            "country": source_cfg["country"],
                            "subreddit": source_cfg.get("subreddit", ""),
                            "feed_name": source_cfg["feed_name"],
                            "source_feed_url": source_cfg["feed_url"],
                            "url": url,
                            "title": clean_text(getattr(entry, "title", "")),
                            "summary": clean_text(getattr(entry, "summary", "") or getattr(entry, "description", "")),
                            "authors": clean_text(getattr(entry, "author", "")),
                            "published_at": published_at,
                        }
                    )
                    self.source_stats[source_name]["accepted"] += 1
            except Exception as exc:
                self.source_stats[source_name]["errors"] += 1
                self.logger.error("Feed error (%s): %s", source_cfg["feed_name"], exc)

        candidates.sort(key=lambda x: x.get("published_at") or self.active_start)
        return candidates[:MAX_ITEMS_PER_WINDOW]

    def _extract(self, item: Dict[str, Any]) -> Dict[str, Any]:
        if item["source_type"] == "reddit":
            item["body"] = item["summary"]
            item["domain"] = domain_from_url(item["url"]) or "reddit.com"
            return item

        body = ""
        try:
            resp = self.session.get(item["url"], timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200 and resp.text:
                raw = trafilatura.extract(resp.text, output_format="json", with_metadata=True, url=item["url"])
                if raw:
                    parsed = json.loads(raw)
                    body = clean_text(parsed.get("text", ""))
                    item["title"] = clean_text(parsed.get("title", "")) or item["title"]
                    item["authors"] = clean_text(parsed.get("author", "")) or item["authors"]
                    item["summary"] = item["summary"] or clean_text(parsed.get("description", ""))
                if not body:
                    soup = BeautifulSoup(resp.text, "html.parser")
                    body = " ".join(clean_text(p.get_text(" ", strip=True)) for p in soup.find_all("p")[:40])
        except Exception:
            pass

        item["body"] = clean_text(body)
        item["domain"] = domain_from_url(item["url"])
        return item

    def _topics(self, text: str, subreddit: str) -> List[str]:
        lowered = text.lower()
        topics = [t for t, kws in TOPIC_RULES.items() if any(k in lowered for k in kws)]
        if subreddit in {"stocks"} and "markets" not in topics:
            topics.insert(0, "markets")
        return topics or ["general"]

    def _canonical_id(self, item: Dict[str, Any]) -> str:
        base = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", item["title"].lower())).strip()
        return hash_text(base or item["url"])

    def _process_item(self, item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        if item["url"] in self.buffer_urls:
            return None
        item = self._extract(item)
        text = " ".join([item["title"], item["summary"], item.get("body", "")])
        topics = self._topics(text, item.get("subreddit", ""))
        sent = self.sentiment.score(text, topics, source_type=item.get("source_type", "news"))

        out = {
            **item,
            "published_at": item["published_at"].isoformat() if isinstance(item.get("published_at"), datetime) else None,
            "first_seen_at": amsterdam_now().isoformat(),
            "origin_country": item["country"] if item["source_type"] == "news" else "Global/Unknown",
            "topics": topics,
            "canonical_id": self._canonical_id(item),
            "final_sentiment_label": sent["label"],
            "final_sentiment_score": sent["raw"],
            "market_sentiment_label": sent["market_label"],
            "market_sentiment_score": sent["market_raw"],
            "sentiment_model_used": sent["model_used"],
            "sentiment_model_note": sent["model_note"],
            "confidence_score": sent["confidence"],
        }
        self.buffer_urls.add(item["url"])
        self.state.seen_urls[item["url"]] = amsterdam_now().isoformat()
        return out

    def _finalize(self) -> Dict[str, Any]:
        df = pd.DataFrame([asdict(ArticleRecord(
            article_id=hash_text(it["url"]),
            canonical_id=it["canonical_id"],
            source_type=it["source_type"],
            publisher=it["publisher"],
            country=it["country"],
            origin_country=it["origin_country"],
            feed_name=it["feed_name"],
            subreddit=it.get("subreddit", ""),
            source_feed_url=it["source_feed_url"],
            url=it["url"],
            domain=it.get("domain", ""),
            title=it.get("title", ""),
            summary=it.get("summary", ""),
            body=it.get("body", ""),
            authors=it.get("authors", ""),
            published_at=it.get("published_at"),
            first_seen_at=it["first_seen_at"],
            final_sentiment_label=it["final_sentiment_label"],
            final_sentiment_score=it["final_sentiment_score"],
            market_sentiment_label=it["market_sentiment_label"],
            market_sentiment_score=it["market_sentiment_score"],
            sentiment_model_used=it.get("sentiment_model_used", "unknown"),
            sentiment_model_note=it.get("sentiment_model_note", ""),
            confidence_score=it["confidence_score"],
            topics=it["topics"],
        )) for it in self.buffer])

        summary = {
            "window_start": self.active_start.isoformat(),
            "window_end": self.active_end.isoformat(),
            "article_count": int(len(df)),
            "canonical_article_count": int(df["canonical_id"].nunique()) if not df.empty else 0,
            "avg_market_sentiment": float(df["market_sentiment_score"].mean()) if not df.empty else None,
            "source_stats": self.source_stats,
            "positive_share": float((df["market_sentiment_label"] == "positive").mean()) if not df.empty else 0.0,
            "negative_share": float((df["market_sentiment_label"] == "negative").mean()) if not df.empty else 0.0,
            "neutral_share": float((df["market_sentiment_label"] == "neutral").mean()) if not df.empty else 0.0,
        }
        prediction_block = self._build_prediction_block(df, cycle="hourly", window_end=self.active_end)
        summary["prediction_block"] = prediction_block
        transparency_rows = (
            df[["title", "source_type", "final_sentiment_label", "market_sentiment_label", "sentiment_model_used", "sentiment_model_note"]]
            .fillna("")
            .to_dict(orient="records")
            if not df.empty else []
        )

        stub = f"Report_{self.active_end.strftime('%Y-%m-%d')}_{self.active_end.strftime('%H')}"
        reports, articles = self._day_dirs(self.active_end)

        if not df.empty:
            write_df = df.copy()
            write_df["topics"] = write_df["topics"].apply(lambda x: ", ".join(x))
            write_df.to_csv(articles / f"{stub}.csv", index=False)
        else:
            pd.DataFrame().to_csv(articles / f"{stub}.csv", index=False)

        (reports / f"{stub}.html").write_text(
            self._build_hourly_dashboard_html(stub, summary, prediction_block, transparency_rows),
            encoding="utf-8",
        )
        self._write_24h_report(self.active_end)

        self.state.completed_windows.add(self.active_end.isoformat())
        self.state.run_history.append(summary)
        self.state.persist()

        self.logger.info("Window finalized: %s", stub)
        self.buffer = []
        self.buffer_urls = set()
        self.source_stats = {}
        self.active_start = self.active_end
        self.active_end = self.active_start + timedelta(hours=1)
        gc.collect()
        return summary

    def _parse_window_end_from_stub(self, stub: str) -> Optional[datetime]:
        # Expected: Report_YYYY-MM-DD_HH
        m = re.match(r"^Report_(\d{4}-\d{2}-\d{2})_(\d{2})$", stub)
        if not m:
            return None
        try:
            dt = datetime.fromisoformat(f"{m.group(1)}T{m.group(2)}:00:00")
            return dt.replace(tzinfo=AMSTERDAM_TZ)
        except Exception:
            return None

    def _write_24h_report(self, window_end: datetime) -> None:
        reports_dir, articles_dir = self._day_dirs(window_end)
        start_24h = window_end - timedelta(hours=24)
        frames: List[pd.DataFrame] = []

        for csv_file in (self.output_root / "Data").glob("**/Report_*.csv"):
            stub = csv_file.stem
            end_dt = self._parse_window_end_from_stub(stub)
            if end_dt is None:
                continue
            if not (start_24h < end_dt <= window_end):
                continue
            try:
                df = pd.read_csv(csv_file)
                if not df.empty:
                    frames.append(df)
            except Exception:
                continue

        if frames:
            df24 = pd.concat(frames, ignore_index=True)
            if "canonical_id" in df24.columns:
                canonical_count = int(df24["canonical_id"].nunique())
            else:
                canonical_count = int(len(df24))
            avg_market = float(df24["market_sentiment_score"].mean()) if "market_sentiment_score" in df24.columns else None
            pos_share = float((df24["market_sentiment_label"] == "positive").mean()) if "market_sentiment_label" in df24.columns else 0.0
            neg_share = float((df24["market_sentiment_label"] == "negative").mean()) if "market_sentiment_label" in df24.columns else 0.0
            neu_share = float((df24["market_sentiment_label"] == "neutral").mean()) if "market_sentiment_label" in df24.columns else 0.0
        else:
            df24 = pd.DataFrame()
            canonical_count = 0
            avg_market = None
            pos_share = 0.0
            neg_share = 0.0
            neu_share = 0.0

        summary24 = {
            "window_type": "rolling_24h",
            "window_start": start_24h.isoformat(),
            "window_end": window_end.isoformat(),
            "article_count": int(len(df24)),
            "canonical_article_count": canonical_count,
            "avg_market_sentiment": avg_market,
            "positive_share": pos_share,
            "negative_share": neg_share,
            "neutral_share": neu_share,
        }
        prediction_block = self._build_prediction_block(df24, cycle="24h", window_end=window_end)
        summary24["prediction_block"] = prediction_block
        transparency_rows = (
            df24[["title", "source_type", "final_sentiment_label", "market_sentiment_label", "sentiment_model_used", "sentiment_model_note"]]
            .fillna("")
            .to_dict(orient="records")
            if not df24.empty and all(c in df24.columns for c in ["title", "source_type", "final_sentiment_label", "market_sentiment_label", "sentiment_model_used", "sentiment_model_note"])
            else []
        )

        stub24 = f"Report_24h_{window_end.strftime('%Y-%m-%d')}_{window_end.strftime('%H')}"
        (reports_dir / f"{stub24}.html").write_text(
            self._build_24h_dashboard_html(stub24, summary24, prediction_block, transparency_rows),
            encoding="utf-8",
        )

    def _fetch_prices(self, tickers: List[str]) -> Dict[str, Optional[float]]:
        if not tickers:
            return {}
        try:
            symbols = ",".join(tickers)
            url = f"https://query1.finance.yahoo.com/v7/finance/quote?symbols={symbols}"
            resp = self.session.get(url, timeout=10)
            if resp.status_code != 200:
                return {t: None for t in tickers}
            payload = resp.json()
            out = {t: None for t in tickers}
            for row in payload.get("quoteResponse", {}).get("result", []):
                sym = row.get("symbol")
                px = row.get("regularMarketPrice")
                if sym in out and isinstance(px, (int, float)):
                    out[sym] = float(px)
            return out
        except Exception:
            return {t: None for t in tickers}

    def _build_prediction_block(self, df: pd.DataFrame, cycle: str, window_end: datetime) -> Dict[str, Any]:
        state_slot = self.state.hourly_predictions if cycle == "hourly" else self.state.daily_predictions
        if df.empty:
            return {"cycle": cycle, "generated_at": window_end.isoformat(), "sectors": [], "evaluation": []}

        sector_scores: Dict[str, List[float]] = {}
        for _, row in df.iterrows():
            raw_topics = row.get("topics", [])
            if isinstance(raw_topics, str):
                raw_topics = [t.strip() for t in raw_topics.split(",") if t.strip()]
            for topic in raw_topics:
                sector = SECTOR_TOPIC_MAP.get(topic)
                if sector:
                    sector_scores.setdefault(sector, []).append(float(row.get("market_sentiment_score", 0.0)))

        if not sector_scores:
            return {"cycle": cycle, "generated_at": window_end.isoformat(), "sectors": [], "evaluation": []}

        ranked = sorted(
            ((s, float(np.mean(vals))) for s, vals in sector_scores.items() if vals),
            key=lambda x: x[1],
            reverse=True,
        )
        chosen: List[Tuple[str, str, float]] = []
        chosen.append((ranked[0][0], "outperform", ranked[0][1]))
        if len(ranked) > 1:
            chosen.append((ranked[-1][0], "underperform", ranked[-1][1]))

        sector_rows = []
        current_tickers: List[str] = []
        for sector, direction, score in chosen:
            names = SECTOR_TICKERS.get(sector, [])[:2]
            firms = [{"ticker": t, "firm": n, "direction": direction} for t, n in names]
            current_tickers.extend([f["ticker"] for f in firms])
            sector_rows.append({"sector": sector, "sector_score": score, "direction": direction, "firms": firms})

        current_prices = self._fetch_prices(current_tickers)
        for row in sector_rows:
            for firm in row["firms"]:
                firm["price_now"] = current_prices.get(firm["ticker"])

        evaluation = []
        prev = state_slot.get("last_prediction")
        if prev and prev.get("firms"):
            prev_tickers = [f["ticker"] for f in prev["firms"]]
            latest_prices = self._fetch_prices(prev_tickers)
            for firm in prev["firms"]:
                old_price = firm.get("price_at_prediction")
                new_price = latest_prices.get(firm["ticker"])
                if old_price is None or new_price is None:
                    continue
                move = "up" if new_price > old_price else ("down" if new_price < old_price else "flat")
                correct = (firm.get("direction") == "outperform" and move == "up") or (firm.get("direction") == "underperform" and move == "down")
                evaluation.append(
                    {
                        "ticker": firm["ticker"],
                        "firm": firm.get("firm"),
                        "direction_predicted": firm.get("direction"),
                        "price_previous": old_price,
                        "price_now": new_price,
                        "move": move,
                        "correct": bool(correct),
                    }
                )

        flat_current = []
        for row in sector_rows:
            for firm in row["firms"]:
                flat_current.append(
                    {
                        "ticker": firm["ticker"],
                        "firm": firm["firm"],
                        "direction": firm["direction"],
                        "price_at_prediction": firm.get("price_now"),
                        "sector": row["sector"],
                    }
                )
        state_slot["last_prediction"] = {"generated_at": window_end.isoformat(), "firms": flat_current}
        return {"cycle": cycle, "generated_at": window_end.isoformat(), "sectors": sector_rows, "evaluation": evaluation}

    def _build_prediction_html(self, block: Dict[str, Any]) -> str:
        if not block.get("sectors"):
            return "<h2>Prediction Tracker</h2><p>No sector signal available for this cycle.</p>"
        lines = ["<h2>Prediction Tracker</h2>", "<h3>Current picks (next cycle)</h3>", "<ul>"]
        for sector in block["sectors"]:
            for firm in sector["firms"]:
                lines.append(
                    f"<li>{html.escape(sector['sector'])} | {html.escape(firm['firm'])} ({html.escape(firm['ticker'])}) "
                    f"| expected {html.escape(firm['direction'])} | price now={firm.get('price_now')}</li>"
                )
        lines.append("</ul>")
        if block.get("evaluation"):
            lines.append("<h3>Previous-cycle prediction check</h3><ul>")
            for row in block["evaluation"]:
                lines.append(
                    f"<li>{html.escape(row['firm'])} ({html.escape(row['ticker'])}) | predicted={html.escape(row['direction_predicted'])} | "
                    f"prev={row['price_previous']} now={row['price_now']} move={html.escape(row['move'])} | correct={row['correct']}</li>"
                )
            lines.append("</ul>")
        else:
            lines.append("<p>No previous price memory available yet for evaluation.</p>")
        return "".join(lines)

    def _build_transparency_html(self, rows: List[Dict[str, Any]]) -> str:
        if not rows:
            return "<h2>Sentiment Modeling Transparency</h2><p>No items in this cycle.</p>"
        lines = ["<h2>Sentiment Modeling Transparency (per item)</h2><ul>"]
        for r in rows:
            lines.append(
                f"<li>{html.escape(str(r.get('title', '')))} | source={html.escape(str(r.get('source_type', '')))} | "
                f"text={html.escape(str(r.get('final_sentiment_label', '')))} | market={html.escape(str(r.get('market_sentiment_label', '')))} | "
                f"model={html.escape(str(r.get('sentiment_model_used', '')))} | note={html.escape(str(r.get('sentiment_model_note', '')))}</li>"
            )
        lines.append("</ul>")
        return "".join(lines)

    def _build_hourly_dashboard_html(self, stub: str, summary: Dict[str, Any], prediction_block: Dict[str, Any], transparency_rows: List[Dict[str, Any]]) -> str:
        pred = self._build_prediction_html(prediction_block)
        transparency = self._build_transparency_html(transparency_rows)
        return (
            f"<html><body><h1>{html.escape(stub)}</h1>{pred}"
            f"<h2>Hourly Summary</h2><p>Items: {summary['article_count']}</p>"
            f"<p>Canonical: {summary['canonical_article_count']}</p>"
            f"<p>Avg market sentiment: {summary['avg_market_sentiment']}</p>"
            f"<p>Positive: {summary['positive_share']:.2%} | Negative: {summary['negative_share']:.2%} | Neutral: {summary['neutral_share']:.2%}</p>"
            f"{transparency}"
            f"</body></html>"
        )

    def _build_24h_dashboard_html(self, stub: str, summary: Dict[str, Any], prediction_block: Dict[str, Any], transparency_rows: List[Dict[str, Any]]) -> str:
        pred = self._build_prediction_html(prediction_block)
        transparency = self._build_transparency_html(transparency_rows)
        return (
            f"<html><body><h1>{html.escape(stub)}</h1>{pred}"
            f"<h2>Rolling 24h Summary</h2><p>Window: {html.escape(summary['window_start'])} → {html.escape(summary['window_end'])}</p>"
            f"<p>Items: {summary['article_count']}</p><p>Canonical: {summary['canonical_article_count']}</p>"
            f"<p>Avg market sentiment: {summary['avg_market_sentiment']}</p>"
            f"<p>Positive: {summary['positive_share']:.2%} | Negative: {summary['negative_share']:.2%} | Neutral: {summary['neutral_share']:.2%}</p>"
            f"{transparency}"
            f"</body></html>"
        )

    def run_forever(self, run_once: bool = False) -> None:
        self.logger.info("Started scraper. Active window: %s -> %s", self.active_start.isoformat(), self.active_end.isoformat())
        while True:
            try:
                if amsterdam_now() >= self.active_end and self.active_end.isoformat() not in self.state.completed_windows:
                    self._finalize()
                    if run_once:
                        return
                    continue

                for item in self._discover():
                    try:
                        processed = self._process_item(item)
                        if processed:
                            self.buffer.append(processed)
                    except Exception as exc:
                        self.logger.error("Item processing error: %s", exc)
                        self.logger.error(traceback.format_exc())

                sleep_for = min(POLL_INTERVAL_SECONDS, max(1, int((self.active_end - amsterdam_now()).total_seconds())))
                if run_once:
                    self._finalize()
                    return
                time.sleep(sleep_for)
            except KeyboardInterrupt:
                self.logger.info("Stopped by user.")
                self.state.persist()
                break
            except Exception as exc:
                self.logger.error("Loop failure: %s", exc)
                self.logger.error(traceback.format_exc())
                self.state.persist()
                time.sleep(20)


def default_sources() -> List[FeedSource]:
    return [FeedSource(**cfg) for cfg in SOURCE_FEEDS + REDDIT_SOURCES]


def load_sources_from_file(path: Optional[str], logger: Optional[logging.Logger] = None) -> List[FeedSource]:
    if not path:
        return default_sources()
    p = Path(path)
    if not p.exists():
        if logger:
            logger.warning("Feeds file not found: %s. Falling back to defaults.", path)
        return default_sources()
    try:
        payload = json.loads(p.read_text(encoding="utf-8"))
        if not isinstance(payload, list):
            raise ValueError("Feeds file must contain a list of sources")
        sources: List[FeedSource] = []
        for row in payload:
            if not isinstance(row, dict):
                continue
            if row.get("source_type") not in {"news", "reddit"}:
                continue
            if not row.get("feed_url", "").startswith(("http://", "https://")):
                continue
            sources.append(
                FeedSource(
                    publisher=row.get("publisher", "Unknown"),
                    country=row.get("country", "Unknown"),
                    feed_name=row.get("feed_name", row.get("publisher", "feed")),
                    feed_url=row["feed_url"],
                    source_type=row["source_type"],
                    subreddit=row.get("subreddit", ""),
                )
            )
        return sources or default_sources()
    except Exception:
        if logger:
            logger.warning("Could not parse feeds file: %s. Falling back to defaults.", path)
        return default_sources()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Continuous news + Reddit sentiment scraper")
    parser.add_argument("--run-once", action="store_true", help="Collect current window once and finalize immediately")
    parser.add_argument("--feeds-file", default=None, help="Optional JSON file with custom feeds list")
    args, _ = parser.parse_known_args()

    maybe_mount_drive()
    output_root = default_output_root()
    sources = load_sources_from_file(args.feeds_file)
    scraper = Scraper(output_root, sources=sources)
    scraper.run_forever(run_once=args.run_once)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

INFO:news_sentiment_scraper:Started scraper. Active window: 2026-04-08T16:00:00+02:00 -> 2026-04-08T17:00:00+02:00


2026-04-08 17:00:06 CEST | INFO | Window finalized: Report_2026-04-08_17


INFO:news_sentiment_scraper:Window finalized: Report_2026-04-08_17


2026-04-08 17:03:48 CEST | INFO | Stopped by user.


INFO:news_sentiment_scraper:Stopped by user.
